<a href="https://colab.research.google.com/github/maggiecrowner/DS5001-Final-Project/blob/main/CORPUS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Parsed and Annotated Data

In [1]:
! git clone https://github.com/maggiecrowner/DS5001-Final-Project.git

Cloning into 'DS5001-Final-Project'...
remote: Enumerating objects: 260, done.
remote: Counting objects: 100% (85/85), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 260 (delta 45), reused 4 (delta 4), pack-reused 175 (from 1)
Receiving objects: 100% (260/260), 15.04 MiB | 15.79 MiB/s, done.
Resolving deltas: 100% (96/96), done.


In [2]:
import pandas as pd
import io
import requests

box_files = {
    "PostMalone.csv":    "https://virginia.box.com/shared/static/o6v4vb3c3pe7kwysmbho7e8agi1co4z4.csv",
    "TaylorSwift.csv":   "https://virginia.box.com/shared/static/ynww2btxjuvl1cei9yaawxm0jgkrqvfs.csv",
    "EdSheeran.csv":     "https://virginia.box.com/shared/static/6zgbpge5yklaiiyypq62eif15axxrm07.csv",
    "JustinBieber.csv":  "https://virginia.box.com/shared/static/k92hby4d8pcbqc2ka9kpeicu4wv0jaiw.csv",
    "ColdPlay.csv":      "https://virginia.box.com/shared/static/2mb65mgblh008fg5s6sfio3bdey1eiph.csv",
    "Maroon5.csv":       "https://virginia.box.com/shared/static/v0az2kge2s8lycapyvzsmx813bb6avkt.csv",
    "Beyonce.csv":       "https://virginia.box.com/shared/static/3giiurqlx8jrs1t83gx0fisstxzv0wy8.csv",
    "LadyGaga.csv":      "https://virginia.box.com/shared/static/uzwtc77cxjauk2vn5xoeqms50prw5of9.csv",
    "BillieEilish.csv":  "https://virginia.box.com/shared/static/airnc5s6mw6c3zrieqps4e3nxc0stk0m.csv",
    "ArianaGrande.csv":  "https://virginia.box.com/shared/static/8m1ekqm8l303le447r3vzgpn773xo9pv.csv",
}

dfs = []
for filename, url in box_files.items():
    df = pd.read_csv(url)
    df['_source_file'] = filename
    dfs.append(df)
    print(f"Loaded {filename}")

source_data = pd.concat(dfs, ignore_index=True)
print(f"\nCorpus shape: {source_data.shape}")
source_data.head()

Loaded PostMalone.csv
Loaded TaylorSwift.csv
Loaded EdSheeran.csv
Loaded JustinBieber.csv
Loaded ColdPlay.csv
Loaded Maroon5.csv
Loaded Beyonce.csv
Loaded LadyGaga.csv
Loaded BillieEilish.csv
Loaded ArianaGrande.csv

Corpus shape: (3073, 8)


,Unnamed: 0,Artist,Title,Album,Year,Date,Lyric,_source_file
0,0.0,Post Malone,​​rockstar,beerbongs & bentleys,2017.0,2017-09-15,post malone hahahahaha tank god ayy ayy post...,PostMalone.csv
1,1.0,Post Malone,White Iverson,Stoney (Deluxe),2015.0,2015-02-04,double ot i'm a new three saucin' saucin' i'...,PostMalone.csv
2,2.0,Post Malone,Congratulations,Stoney (Deluxe),2016.0,2016-11-04,post malone mmmmm yeah yeah mmmmm yeah hey p...,PostMalone.csv
3,3.0,Post Malone,Psycho,beerbongs & bentleys,2018.0,2018-02-23,post malone damn my ap goin' psycho lil' mama ...,PostMalone.csv
4,4.0,Post Malone,I Fall Apart,Stoney (Deluxe),2016.0,2016-12-09,ooh i fall apart ooh yeah mmm yeah she told ...,PostMalone.csv


In [3]:
source_data = source_data[~source_data['Lyric'].str.contains('lyrics for this song have yet to be released please check back once the song has been released', case=False, na=False)]
print(source_data.shape)

(2964, 8)


## CORPUS

In [4]:
import nltk
import re
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('punkt')
nltk.download('punkt_tab')

pos_group_map = {
    'NN': 'NOUN', 'NNS': 'NOUN', 'NNP': 'NOUN', 'NNPS': 'NOUN',
    'VB': 'VERB', 'VBD': 'VERB', 'VBG': 'VERB', 'VBN': 'VERB', 'VBP': 'VERB', 'VBZ': 'VERB',
    'JJ': 'ADJ',  'JJR': 'ADJ',  'JJS': 'ADJ',
    'RB': 'ADV',  'RBR': 'ADV',  'RBS': 'ADV',
}

records = []
for _, row in source_data.iterrows():
    artist = row['Artist']
    album  = row['Album']
    track  = row['Title']
    lyric  = str(row['Lyric'])

    words    = lyric.split()
    pos_tags = nltk.pos_tag(words)

    for word_id, (token_str, pos) in enumerate(pos_tags):
        term_str  = re.sub(r'[^\w\s]', '', token_str).lower()
        pos_group = pos_group_map.get(pos, 'OTHER')

        records.append({
            'Artist':    artist,
            'Album':     album,
            'Title':     track,
            'WordID':    word_id,
            'token_str': token_str,
            'term_str':  term_str,
            'pos':       pos,
            'pos_group': pos_group,
        })

CORPUS = pd.DataFrame(records)
CORPUS = CORPUS.set_index(['Artist', 'Album', 'Title', 'WordID'])
print(f" Corpus shape: {CORPUS.shape}")
CORPUS.head()

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


 Corpus shape: (985424, 4)


token_str    term_str  \
Artist      Album                Title      WordID                           
Post Malone beerbongs & bentleys ​​rockstar 0             post        post   
                                            1           malone      malone   
                                            2       hahahahaha  hahahahaha   
                                            3             tank        tank   
                                            4              god         god   

                                                   pos pos_group  
Artist      Album                Title      WordID                
Post Malone beerbongs & bentleys ​​rockstar 0       NN      NOUN  
                                            1       NN      NOUN  
                                            2       NN      NOUN  
                                            3       NN      NOUN  
                                            4       NN      NOUN

In [5]:
CORPUS.to_csv('/content/DS5001-Final-Project/CORPUS.csv', sep='|', index=True)

In [7]:
print("Number of observations/tokens:", len(CORPUS['token_str']))

Number of observations/tokens: 985424
